# Converting labels from the Planet PNG to the Sentinel-2 PNG

We have the labels for the Planet images, and we need to convert them to geographic coordinates, and then to the dimensions of the Sentinel-2 images. 

In [27]:
import json
import os
import pandas as pd
import numpy as np
import rasterio

from PIL import Image
from pyproj import Transformer   
from datetime import datetime    

## Set the names of the files

All that should need to be changed are the directories, if they are different to the structure here, and the names of the files. The dates of the Planet data should be similar to the Sentinel-2 data, so we can use the same date for the Sentinel-2 data.

In [142]:
# Base directory for the images
base_dir = 'images/planet_2_sentinel'
# Base directory for the Planet tif image
planet_tif_dir = f'{base_dir}/planet_tifs'
# Base directory for the png image
planet_png_dir = f'{base_dir}/planet_pngs'

# Base directory for the Sentinel tif image
S2_tif_dir = f'{base_dir}/S2_tifs'
# Base directory for the png image
S2_png_dir = f'{base_dir}/S2_pngs'

# Planet tif image name - should be the same for the png and the png so this shouldn't need to be changed
planet_tif_name = '20240604_mimal_test'
# Sentinel tif image name - should also be the same
S2_tif_name = '2024-06_mimal_test_S2'

## Import the Planet raster file to get the metadata

We need the information from the raster as the json file needs to be in reference to the origin and transformation of the tif file.

In [143]:
planet_tif_path = os.path.join(planet_tif_dir, planet_tif_name + '.tif')

with rasterio.open(planet_tif_path) as raster:
    # Read the raster band
    planet_raster = raster.read(1)
    # Get the metadata of the raster
    planet_raster_meta = raster.meta
    # Get the raster transform parameters
    planet_raster_transform = raster.transform

print("Shape of the raster (rows, columns):")
print(planet_raster.shape)
print("\n")

print("Raster metadata:")
print(planet_raster_meta)
print("\n")

print("Affine transformation parameters:")
print(planet_raster_transform)

Shape of the raster (rows, columns):
(11309, 14180)


Raster metadata:
{'driver': 'GTiff', 'dtype': 'uint16', 'nodata': 0.0, 'width': 14180, 'height': 11309, 'count': 4, 'crs': CRS.from_epsg(32753), 'transform': Affine(3.0, 0.0, 438081.0,
       0.0, -3.0, 8533512.0)}


Affine transformation parameters:
| 3.00, 0.00, 438081.00|
| 0.00,-3.00, 8533512.00|
| 0.00, 0.00, 1.00|


## Account for the png padding

As the png has padding, we need to figure out how much to add to the x and y coordinates to get the correct position for the labels in the png file.

This function will be different if there is padding on both sides and the top and bottom of the image. I've just accounted for padding on the left and top of the image by adding the difference in between the png size and the raster size to the x (difference in width) and y (difference in height) coordinates.

If the padding is consistent (e.g. tile_size - stride), we can just add that padding to the x and y coordinates. In that case, set width_diff and height_diff to the padding amount for the x and y axes respectively.

In [144]:
Image.MAX_IMAGE_PIXELS = 933120000 # change to greater than the nnumber of pixels (only needed if there's a warning)

# Load the png image
planet_image = Image.open(f'{planet_png_dir}/{planet_tif_name}.png')

# Convert the image to a numpy array
pixel_data = np.array(planet_image)
print(f'Shape of the image (includes padding) (rows, columns, bands): {pixel_data.shape}')

Shape of the image (includes padding) (rows, columns, bands): (12064, 14976, 3)


### Get Planet image information

In [145]:
# Get image information
planet_png_width, planet_png_height = planet_image.size
print(f'Planet image size: ({planet_png_width}, {planet_png_height})')

# Calculate the different in the width and height of the image and the raster
planet_width_diff = planet_png_width - planet_raster.shape[1]
planet_height_diff = planet_png_height - planet_raster.shape[0]
print(f'Difference in width: {planet_width_diff}')
print(f'Difference in height: {planet_height_diff}')


# if using images with padded left and right (and top and bottom)
tile_size = 416
stride = 104
min_pad = tile_size - stride
# subtract the padding on the right and bottom from the differences
# if there is no padding on the right and bottom, comment out the following two lines
planet_width_diff = planet_width_diff - min_pad
planet_height_diff = planet_height_diff - min_pad

print(f'Difference in width after removing right padding: {planet_width_diff}')
print(f'Difference in height after removing bottom padding: {planet_height_diff}')

Planet image size: (14976, 12064)
Difference in width: 796
Difference in height: 755
Difference in width after removing right padding: 484
Difference in height after removing bottom padding: 443


# Import the labels for the Planet PNG

In [146]:
# Path to the LabelMe JSON file
labelme_json_path = os.path.join(planet_png_dir, f'{planet_tif_name}.json')

# Load the JSON file
with open(labelme_json_path, 'r') as f:
    labelme_data = json.load(f)

# Extract bounding box labels
bounding_boxes = []
for shape in labelme_data['shapes']:
    label = shape['label']
    points = shape['points']
    x_min, y_min = points[0]
    x_max, y_max = points[1]
    bounding_boxes.append({'label': label, 'x_min': x_min, 'y_min': y_min, 'x_max': x_max, 'y_max': y_max})

# Print the bounding boxes
# print(bounding_boxes)

# Count the number of bounding boxes
print(f'Number of bounding boxes: {len(bounding_boxes)}')

Number of bounding boxes: 187


In [147]:
def convert_pixel_to_geo_coords(bboxes, raster_transform, width_diff, height_diff):
    """
    Convert pixel coordinates from bounding boxes to geographic coordinates.
    
    Args:
        bboxes (list): List of dictionaries containing bounding box coordinates
        raster_transform (affine.Affine): Affine transformation object
        width_diff (int): Padding on the left side of the image
        height_diff (int): Padding on the top side of the image
        
    Returns:
        list: List of dictionaries with bounding boxes in geographic coordinates
    """
    geo_boxes = []
    
    for bbox in bboxes:
        
        # Remove the padding from the pixel coordinates
        x_min_px = bbox['x_min'] - width_diff
        y_min_px = bbox['y_min'] - height_diff
        x_max_px = bbox['x_max'] - width_diff
        y_max_px = bbox['y_max'] - height_diff
        
        # Convert pixel coordinates to geographic coordinates using the raster transform
        # The * operator applies the transform to the (x, y) coordinates
        x_min_geo, y_min_geo = raster_transform * (x_min_px, y_min_px)
        x_max_geo, y_max_geo = raster_transform * (x_max_px, y_max_px)
        
        # Create a new dictionary with the geographic coordinates
        geo_box = {
            'label': bbox['label'],
            'x_min_geo': x_min_geo,
            'y_min_geo': y_min_geo,
            'x_max_geo': x_max_geo,
            'y_max_geo': y_max_geo,
            # Keep the original pixel coordinates for reference
            'x_min_px': bbox['x_min'],
            'y_min_px': bbox['y_min'],
            'x_max_px': bbox['x_max'],
            'y_max_px': bbox['y_max']
        }
        
        geo_boxes.append(geo_box)
    
    return geo_boxes

## Convert the pixel coordinates to geographic coordinates

In [148]:
# Convert all bounding boxes to geographic coordinates
geo_coordinates = convert_pixel_to_geo_coords(bounding_boxes, planet_raster_transform, planet_width_diff, planet_height_diff)

# Display the first 5 converted coordinates
for i, box in enumerate(geo_coordinates[:5]):
    print(f"Box {i+1} - Label: {box['label']}")
    print(f"  Pixel: ({box['x_min_px']}, {box['y_min_px']}) to ({box['x_max_px']}, {box['y_max_px']})")
    print(f"  Geo: ({box['x_min_geo']}, {box['y_min_geo']}) to ({box['x_max_geo']}, {box['y_max_geo']})")
    print("")

Box 1 - Label: WH_wet
  Pixel: (2973.1895223420647, 10518.181818181818) to (3030.0462249614793, 10563.328197226501)
  Geo: (445548.5685670262, 8503286.454545455) to (445719.13867488445, 8503151.01540832)

Box 2 - Label: Dry_WH
  Pixel: (2431.790322580645, 10974.935483870968) to (2495.967741935484, 11007.258064516129)
  Geo: (443924.37096774194, 8501916.193548387) to (444116.9032258064, 8501819.225806452)

Box 3 - Label: WH_swamp
  Pixel: (3335.075091575092, 9852.563186813186) to (3575.1428571428573, 9965.714285714284)
  Geo: (446634.2252747253, 8505283.31043956) to (447354.4285714286, 8504943.857142856)

Box 4 - Label: WH_swamp
  Pixel: (5316.9473684210525, 4770.742690058479) to (5758.000000000001, 5048.181818181818)
  Geo: (452579.84210526315, 8520528.771929825) to (453903.0, 8519696.454545455)

Box 5 - Label: WH_swamp
  Pixel: (5328.327102803739, 4290.03313508921) to (5655.272727272728, 4509.090909090909)
  Geo: (452613.9813084112, 8521970.900594732) to (453594.8181818182, 8521313.72

## Import the Sentinel-2 raster file to get the metadata

We need the information from the raster as the json file needs to be in reference to the origin and transformation of the tif file.

In [149]:
S2_tif_path = os.path.join(S2_tif_dir, S2_tif_name + '.tif')

with rasterio.open(S2_tif_path) as raster:
    # Read the raster band
    S2_raster = raster.read(1)
    # Get the metadata of the raster
    S2_raster_meta = raster.meta
    # Get the raster transform parameters
    S2_raster_transform = raster.transform

print("Shape of the raster (rows, columns):")
print(S2_raster.shape)
print("\n")

print("Raster metadata:")
print(S2_raster_meta)
print("\n")

print("Affine transformation parameters:")
print(S2_raster_transform)

Shape of the raster (rows, columns):
(3422, 4380)


Raster metadata:
{'driver': 'GTiff', 'dtype': 'uint8', 'nodata': None, 'width': 4380, 'height': 3422, 'count': 3, 'crs': CRS.from_epsg(4326), 'transform': Affine(8.983152841195215e-05, 0.0, 134.42767203983848,
       0.0, -8.983152841195215e-05, -13.26479297989409)}


Affine transformation parameters:
| 0.00, 0.00, 134.43|
| 0.00,-0.00,-13.26|
| 0.00, 0.00, 1.00|


## Convert the labels in the json file to the same projection as the Planet tif file

As the labels are in pixel coordinates, we need to:
- convert them to the same projection as the Planet tif file - CRS.from_epsg(32753)
- convert them to the projection of the Sentinel-2 data - CRS.from_epsg(4326)
- convert them to the Sentinel-2 pixel coordinates using the raster transformation from the S2 metadata.

In [150]:
def convert_to_sentinel_coords(geo_boxes, S2_raster_transform):
    """
    Convert bounding boxes from UTM coordinates (EPSG:32753) to WGS84 (EPSG:4326)
    and then to Sentinel-2 pixel coordinates.
    
    Args:
        geo_boxes (list): List of dictionaries with bounding box coordinates in UTM (EPSG:32753)
        S2_raster_transform (affine.Affine): Affine transformation of the Sentinel-2 raster
        
    Returns:
        list: List of dictionaries with bounding boxes in Sentinel-2 pixel coordinates
    """
    # Create transformer from Planet CRS (EPSG:32753) to Sentinel-2 CRS (EPSG:4326)
    utm_to_wgs84 = Transformer.from_crs('epsg:32753', 'epsg:4326', always_xy=True)
    
    s2_boxes = []
    
    for box in geo_boxes:
        # Convert UTM coordinates to WGS84
        x_min_wgs84, y_min_wgs84 = utm_to_wgs84.transform(box['x_min_geo'], box['y_min_geo'])
        x_max_wgs84, y_max_wgs84 = utm_to_wgs84.transform(box['x_max_geo'], box['y_max_geo'])
        
        # Convert WGS84 to Sentinel-2 pixel coordinates using the inverse transform
        # The ~ operator inverts the affine transform
        x_min_s2_px, y_min_s2_px = ~S2_raster_transform * (x_min_wgs84, y_min_wgs84)
        x_max_s2_px, y_max_s2_px = ~S2_raster_transform * (x_max_wgs84, y_max_wgs84)
        
        # Create a new dictionary with the Sentinel-2 pixel coordinates
        s2_box = {
            'label': box['label'],

            # Store WGS84 coordinates
            'x_min_wgs84': x_min_wgs84,
            'y_min_wgs84': y_min_wgs84,
            'x_max_wgs84': x_max_wgs84,
            'y_max_wgs84': y_max_wgs84,

            # Store Sentinel-2 pixel coordinates
            'x_min_s2_px': x_min_s2_px,
            'y_min_s2_px': y_min_s2_px,
            'x_max_s2_px': x_max_s2_px,
            'y_max_s2_px': y_max_s2_px,
            
            # Keep original UTM and Planet pixel coordinates
            'x_min_utm': box['x_min_geo'],
            'y_min_utm': box['y_min_geo'],
            'x_max_utm': box['x_max_geo'],
            'y_max_utm': box['y_max_geo'],
            'x_min_planet_px': box['x_min_px'],
            'y_min_planet_px': box['y_min_px'],
            'x_max_planet_px': box['x_max_px'],
            'y_max_planet_px': box['y_max_px']
        }
        
        s2_boxes.append(s2_box)
    
    return s2_boxes

## Run the function to convert the bounding boxes to Sentinel-2 pixel coordinates

In [151]:
# Convert the bounding boxes to Sentinel-2 coordinates
sentinel2_boxes = convert_to_sentinel_coords(geo_coordinates, S2_raster_transform)

# Display the first 5 converted coordinates
for i, box in enumerate(sentinel2_boxes[:5]):
    print(f"Box {i+1} - Label: {box['label']}")
    print(f"  Planet Pixel: ({box['x_min_planet_px']:.1f}, {box['y_min_planet_px']:.1f}) to ({box['x_max_planet_px']:.1f}, {box['y_max_planet_px']:.1f})")
    print(f"  UTM (EPSG:32753): ({box['x_min_utm']:.1f}, {box['y_min_utm']:.1f}) to ({box['x_max_utm']:.1f}, {box['y_max_utm']:.1f})")
    print(f"  WGS84 (EPSG:4326): ({box['x_min_wgs84']:.6f}, {box['y_min_wgs84']:.6f}) to ({box['x_max_wgs84']:.6f}, {box['y_max_wgs84']:.6f})")
    print(f"  Sentinel-2 Pixel: ({box['x_min_s2_px']:.1f}, {box['y_min_s2_px']:.1f}) to ({box['x_max_s2_px']:.1f}, {box['y_max_s2_px']:.1f})")
    print("")

Box 1 - Label: WH_wet
  Planet Pixel: (2973.2, 10518.2) to (3030.0, 10563.3)
  UTM (EPSG:32753): (445548.6, 8503286.5) to (445719.1, 8503151.0)
  WGS84 (EPSG:4326): (134.496771, -13.538228) to (134.498345, -13.539456)
  Sentinel-2 Pixel: (769.2, 3043.9) to (786.7, 3057.5)

Box 2 - Label: Dry_WH
  Planet Pixel: (2431.8, 10974.9) to (2496.0, 11007.3)
  UTM (EPSG:32753): (443924.4, 8501916.2) to (444116.9, 8501819.2)
  WGS84 (EPSG:4326): (134.481735, -13.550587) to (134.483512, -13.551468)
  Sentinel-2 Pixel: (601.8, 3181.4) to (621.6, 3191.2)

Box 3 - Label: WH_swamp
  Planet Pixel: (3335.1, 9852.6) to (3575.1, 9965.7)
  UTM (EPSG:32753): (446634.2, 8505283.3) to (447354.4, 8504943.9)
  WGS84 (EPSG:4326): (134.506842, -13.520192) to (134.513491, -13.523275)
  Sentinel-2 Pixel: (881.3, 2843.1) to (955.3, 2877.4)

Box 4 - Label: WH_swamp
  Planet Pixel: (5316.9, 4770.7) to (5758.0, 5048.2)
  UTM (EPSG:32753): (452579.8, 8520528.8) to (453903.0, 8519696.5)
  WGS84 (EPSG:4326): (134.562035, 

## Account for the Sentinel-2 png padding

As the png has padding, we need to figure out how much to add to the x and y coordinates to get the correct position for the labels in the png file.

This function will be different if there is padding on both sides and the top and bottom of the image. I've just accounted for padding on the left and top of the image by adding the difference in between the png size and the raster size to the x (difference in width) and y (difference in height) coordinates.

If the padding is consistent (e.g. tile_size - stride), we can just add that padding to the x and y coordinates. In that case, set width_diff and height_diff to the padding amount for the x and y axes respectively.

In [152]:
# Load the png image
S2_image = Image.open(f'{S2_png_dir}/{S2_tif_name}.png')

# Convert the image to a numpy array
pixel_data = np.array(S2_image)
print(pixel_data.shape)

(4160, 5408, 3)


### Get Sentinel-2 image information

In [153]:
# Get image information
S2_png_width, S2_png_height = S2_image.size
print(f'Image size: ({S2_png_width}, {S2_png_height})')

# Calculate the different in the width and height of the image and the raster
S2_width_diff = S2_png_width - S2_raster.shape[1]
S2_height_diff = S2_png_height - S2_raster.shape[0]
print(f'Difference in width: {S2_width_diff}')
print(f'Difference in height: {S2_height_diff}')


# if using images with padded left and right (and top and bottom)
tile_size = 416
stride = 104
min_pad = tile_size - stride
# subtract the padding on the right and bottom from the differences
S2_width_diff = S2_width_diff - min_pad
S2_height_diff = S2_height_diff - min_pad

print(f'Difference in S2 width after removing right padding: {S2_width_diff}')
print(f'Difference in S2 height after removing bottom padding: {S2_height_diff}')

Image size: (5408, 4160)
Difference in width: 1028
Difference in height: 738
Difference in S2 width after removing right padding: 716
Difference in S2 height after removing bottom padding: 426


## Functions to convert the labels to the Sentinel-2 png ans save json files

In [154]:
def save_labelme_json(json_data, filepath):
    """
    Save LabelMe JSON data to a file.
    
    Args:
        json_data (dict): LabelMe JSON data
        filepath (str): Path to save the JSON file
    """
    with open(filepath, 'w') as f:
        json.dump(json_data, f, indent=2)

# Function to convert the Sentinel-2 bounding boxes to LabelMe format
def convert_sentinel_boxes_to_labelme(boxes, image_path, image_height, image_width, width_diff, height_diff):
    """
    Convert Sentinel-2 bounding boxes to LabelMe JSON format.
    
    Args:
        boxes (list): List of dictionaries containing bounding box information
        image_path (str): Path to the corresponding image file
        image_height (int): Height of the image in pixels
        image_width (int): Width of the image in pixels
        width_diff (int): Width padding to add to pixel coordinates
        height_diff (int): Height padding to add to pixel coordinates
        
    Returns:
        dict: LabelMe formatted JSON
    """
    # Initialize LabelMe JSON structure
    labelme_json = {
        "version": "5.0.1",
        "flags": {},
        "shapes": [],
        "imagePath": os.path.basename(image_path),
        "imageData": None,  # LabelMe stores base64 image data here, but we'll leave it empty
        "imageHeight": image_height,
        "imageWidth": image_width
    }
    
    # Process each bounding box
    for box in boxes:
        # Add the padding to the pixel coordinates
        x_min = box['x_min_s2_px'] + width_diff
        y_min = box['y_min_s2_px'] + height_diff
        x_max = box['x_max_s2_px'] + width_diff
        y_max = box['y_max_s2_px'] + height_diff
        
        # Create a shape for the bounding box
        shape = {
            "label": box['label'],
            "points": [
                [float(x_min), float(y_min)],  # Top-left
                [float(x_max), float(y_max)]   # Bottom-right
            ],
            "group_id": None,
            "shape_type": "rectangle",
            "flags": {}
        }
        
        labelme_json['shapes'].append(shape)
    
    # Add creation time
    labelme_json['timeStamp'] = datetime.now().isoformat()
    
    return labelme_json

## Convert the labels to the Sentinel-2 png

In [155]:
# Convert and save the Sentinel-2 bounding boxes
S2_image_path = f'{S2_png_dir}/{S2_tif_name}.png'

# Convert the bounding boxes to LabelMe format
sentinel2_labelme = convert_sentinel_boxes_to_labelme(
    boxes=sentinel2_boxes,
    image_path=S2_image_path,
    image_height=S2_png_height,
    image_width=S2_png_width,
    width_diff=S2_width_diff,
    height_diff=S2_height_diff
)

# Save the LabelMe JSON file
S2_json_path = f'{S2_png_dir}/{S2_tif_name}.json'
save_labelme_json(sentinel2_labelme, S2_json_path)

print(f"Sentinel-2 bounding boxes converted to LabelMe format and saved to {S2_json_path}")
print(f"Number of bounding boxes: {len(sentinel2_boxes)}")

Sentinel-2 bounding boxes converted to LabelMe format and saved to images/planet_2_sentinel/S2_pngs/2024-06_mimal_test_S2.json
Number of bounding boxes: 187
